In [ ]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

In [2]:
!pip install chromadb langchain_community langchain_openai

In [19]:
import os

# Claves necesarias para LangSmith (EU endpoint)
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')
os.environ["LANGCHAIN_PROJECT"] = "Ecommerce-Agent"
os.environ["LANGSMITH_ENDPOINT"] = userdata.get('LANGSMITH_ENDPOINT')

In [ ]:
from google.colab import userdata
userdata.get('LANGSMITH_API_KEY')

In [27]:
import os
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# 🔹 CONFIG Chroma Cloud (TUS DATOS)
CHROMA_CONFIG = {
    'api_key': userdata.get('CHROMA_API_KEY'),
    'tenant': userdata.get('CHROMA_TENANT'),
    'database': 'Amazon_database',
    'collection': 'langchain'
}



In [22]:
from langsmith import Client

client = Client()

# (Opcional) verificar conexión
print("LangSmith conectado ✅:", client.list_projects())


LangSmith conectado ✅: <generator object Client.list_projects at 0x7f19bc492b90>


In [23]:
import os
from google.colab import userdata

# Set the OpenAI API key as an environment variable
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# 🔹 Cliente + Vectorstore
client = chromadb.CloudClient(**{k: v for k, v in CHROMA_CONFIG.items() if k != 'collection'})
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
    client=client,
    collection_name=CHROMA_CONFIG['collection'],
    embedding_function=embeddings
)


In [24]:
def buscar_similar(query, k=5):
    """Busca productos similares"""
    docs = vectorstore.similarity_search(query, k=k)
    print(f"\n🔍 '{query}' - Top {k}:")
    for i, doc in enumerate(docs, 1):
        print(f"  {i}. {doc.page_content[:100]}...")
    return docs

def buscar_con_score(query, k=5):
    """Busca con scores de similitud"""
    docs_scores = vectorstore.similarity_search_with_score(query, k=k)
    print(f"\n📊 '{query}' - Top {k} (score):")
    for i, (doc, score) in enumerate(docs_scores, 1):
        print(f"  {i}. {doc.page_content[:80]}... (score: {score:.3f})")
    return docs_scores

# 🔹 USO
buscar_similar("desodorante")
buscar_con_score("crema hidratante", k=3)


🔍 'desodorante' - Top 5:
  1. Agree deodorant stick

...
  2. Linha Accordes Boticario - Desodorante Antitranspirante Aerosol 75 Gr - (Boticario Accordes Collecti...
  3. Deeoral - Desodorante NATURAL en Spray - Sin Aluminio - Elimina el mal olor corporal (1.01 Oz)

DESO...
  4. avena instituto espanol fresh deodorant roll on anti-transpirant

Avena instituto espanol Fresh frag...
  5. avena instituto espanol fresh deodorant roll on anti-transpirant

Avena instituto espanol Fresh frag...

📊 'crema hidratante' - Top 3 (score):
  1. Lactovit Mousse Creme, Crema Hidratante para la Cara y el Cuerpo Para Piel Norma... (score: 0.829)
  2. Crema Humectante de Manzana (3-Pack) 2.11oz / 60gr

... (score: 0.880)
  3. MARTIDERM CALAMINA PLUS CREMA 75 ML

... (score: 0.883)


[(Document(metadata={'price': '13.57', 'size': '', 'color': '', 'avg_rating': 4.4, 'title': 'Lactovit Mousse Creme, Crema Hidratante para la Cara y el Cuerpo Para Piel Normal a Seca, Nutritiva y Ligera, 8.5 oz (250ml)', 'description': ''}, page_content='Lactovit Mousse Creme, Crema Hidratante para la Cara y el Cuerpo Para Piel Normal a Seca, Nutritiva y Ligera, 8.5 oz (250ml)\n\n'),
  0.829015),
 (Document(metadata={'description': '', 'price': '18.99', 'color': '', 'title': 'Crema Humectante de Manzana (3-Pack) 2.11oz / 60gr', 'size': '', 'avg_rating': 3.8}, page_content='Crema Humectante de Manzana (3-Pack) 2.11oz / 60gr\n\n'),
  0.8796262),
 (Document(metadata={'description': '', 'avg_rating': 4.4, 'price': '19.49', 'title': 'MARTIDERM CALAMINA PLUS CREMA 75 ML', 'size': '', 'color': ''}, page_content='MARTIDERM CALAMINA PLUS CREMA 75 ML\n\n'),
  0.8833984)]

In [25]:
import os
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
import chromadb

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
# 🔹 Chroma Cloud
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = chromadb.CloudClient(
    api_key= userdata.get('CHROMA_API_KEY'),
    tenant= userdata.get('CHROMA_TENANT'),
    database='Amazon_database'
)
vectorstore = Chroma(
    client=client,
    collection_name="langchain",
    embedding_function=embeddings
)
print("✅ Chroma Cloud ready!")

# 🔹 LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 🔹 Retriever (missing line!)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 🔹 Format function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 🔹 Prompt
prompt = ChatPromptTemplate.from_template("""
Eres un asistente de e-commerce.
Tu único objetivo es ayudar a los usuarios con información sobre productos,
precios, valoraciones y recomendaciones del catálogo.

⚠️ No respondas preguntas sobre programación, IA, Python, APIs, código, ni temas técnicos.
Si el usuario pregunta algo fuera de ese contexto, responde exactamente:
"Solo puedo responder preguntas relacionadas con el catálogo y los productos."

Contexto productos:
{context}

Pregunta: {input}

Responde en español, de forma clara y breve:
""")
# 🔹 RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

def rag_amazon(query, filter_kwargs=None):
    """Versión protegida de RAG: combina guard semántico + prompt restrictivo."""

    if filter_kwargs:
        retriever.search_kwargs["filter"] = filter_kwargs

    respuesta = rag_chain.invoke(query)
    return respuesta


✅ Chroma Cloud ready!


In [26]:
# PRUEBAS
rag_amazon("Recomiéndame un desodorante con buen rating y precio menor de 20 euros")
rag_amazon("Mejor crema hidratante")
rag_amazon("Productos para el pelo muy buenos")

'Te recomiendo los siguientes productos para el cabello:\n\n1. **Savile Crema Para Peinar Liso Colagen 300ml**: Ideal para lograr un peinado liso y suave.\n   \n2. **Cirugia Capilat Kera Fruit Tratamiento Profesional 1 litro + Shampoo Antiresiduo**: Un alisador que promete buenos resultados.\n\n3. **Kaba KABA Shampoo de Cebolla + Bio mascarilla Capilar**: Combina el shampoo de cebolla con una mascarilla nutritiva.\n\n4. **Shampoo Suave de Cebolla + Biomascarilla de Frutos**: Sin necesidad de refrigeración, ideal para el cuidado diario.\n\n5. **Queratina Gold Diamond, Cera Fría 1 Litro**: Un tratamiento de queratina que puede ayudar a mejorar la textura del cabello.\n\nEstos productos son altamente valorados y pueden ser una excelente opción para el cuidado de tu cabello.'

In [11]:
rag_amazon("Hazme un script en python para comunicarme contigo")

'Solo puedo responder preguntas relacionadas con el catálogo y los productos.'

In [12]:
# ----------------------------
# 1️⃣ Imports
# ----------------------------
from langchain_core.tools import tool, Tool # Added Tool for explicit wrapping
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# ----------------------------
# 2️⃣ Carrito y datos de envío
# ----------------------------
carrito = {}
direccion_envio = {}

# ----------------------------
# 3️⃣ Tools
# ----------------------------

@tool
def ver_carrito() -> str:
    """Muestra los productos actualmente en el carrito de compras."""
    if (not carrito) or  (len(carrito)==0):
        return "El carrito está vacío."
    return "\n".join([f"{i+1}. {item['title']} - Precio: {item['price']}"
                      for i, item in enumerate(carrito.values())])

@tool
def agregar_producto(title: str, price: float) -> str:
    """Agrega un producto con un título y un precio especificados al carrito de compras."""
    carrito[title] = {"title": title, "price": price}
    return f"Producto '{title}' agregado al carrito."

@tool
def guardar_direccion(nombre: str, email: str, direccion: str) -> str:
    """Guarda la información de la dirección de envío del cliente."""
    direccion_envio.update({"nombre": nombre, "email": email, "direccion": direccion})
    return f"Dirección guardada para {nombre}."

@tool
def pagar_tarjeta(nombre: str, numero: str, mes: int, año: int, cvc: str) -> str:
    """Procesa un pago con tarjeta de crédito para los productos en el carrito."""
    if not carrito:
        return "No hay productos en el carrito para pagar."
    total = sum([float(item['price'].replace('$','')) for item in carrito.values()])
    carrito.clear()
    return f"Pago realizado con tarjeta {numero[-4:]} por un total de ${total}. ¡Gracias {nombre}!"

# Tool de búsqueda de productos con sus características

# List of all tools
tools = [
    ver_carrito,
    agregar_producto,
   # guardar_direccion,
   # pagar_tarjeta,
    Tool.from_function(
        func=rag_amazon,
        name="rag_amazon",
        description="Busca productos en Amazon y devuelve recomendaciones basadas en la consulta del usuario."
    ) # Explicitly wrap rag_amazon as a Tool
]

# ----------------------------
# 4️⃣ Crear el agente
# ----------------------------
chat_model = ChatOpenAI(model="gpt-5.1", temperature=0)

agent = create_agent(
    chat_model,
    tools, # Use the defined tools list
    system_prompt="Eres un asistente de compras. Usa las funciones correctamente según lo que el usuario pida."
)

# ----------------------------
# 5️⃣ Función de chat interactivo
# ----------------------------
from langchain_core.messages import HumanMessage, AIMessage


messages = []  # This list accumulates the full conversation history

def chat_ecommerce():
    print("¡Bienvenido al asistente de compras! Escribe 'salir' para terminar.\n")
    while True:
        entrada = input("Tú: ")
        if entrada.lower() in ["salir", "exit"]:
            print("Agente: Gracias por usar el asistente de compras. ¡Hasta luego!")
            break

        messages.append(HumanMessage(content=entrada))

        result = agent.invoke({"messages": messages})  # ✅ "messages" (plural)

        # ✅ Persist the full updated message history (includes tool calls, responses, etc.)
        messages.clear()
        messages.extend(result["messages"])

        last_ai = result["messages"][-1]
        print(f"Chatbot: {last_ai.content}\n")


In [13]:
chat_ecommerce()

¡Bienvenido al asistente de compras! Escribe 'salir' para terminar.



KeyboardInterrupt: Interrupted by user

In [14]:
#messages.append({"role": "user", "content": "un jabon de glicerina por menos de 10 euros"})
result = agent.invoke({"messages": [{"role": "user", "content": "buscame un jabon de glicerina por menos de 10 euros"}]})
#result["messages"][-1].content

In [15]:
result

{'messages': [HumanMessage(content='buscame un jabon de glicerina por menos de 10 euros', additional_kwargs={}, response_metadata={}, id='6fb1fe76-34be-4173-be69-a9c5b6485c0d'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 234, 'total_tokens': 271, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DG8FIB3qZIjPjLwqEkPcZ7e3q8dB6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cbf60-c4a7-7993-bc67-caddfafc651e-0', tool_calls=[{'name': 'rag_amazon', 'args': {'__arg1': 'jabón de glicerina menos de 10 euros'}, 'id': 'call_KfwpaPSFzwDz57Z8vi1SBWL9', 'type': 'tool_call'}], invalid_tool_calls=[], us

In [17]:
# 1️⃣ Conectar con Google Drive
# ----------------------------
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [18]:
# ----------------------------
# 1️⃣ Imports
# ----------------------------
import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

# Asumimos que ya tienes tu objeto `agent` creado previamente
# y el resto de tu entorno (tools, modelo, etc.)

# ----------------------------
# 2️⃣ Inicializar el historial de conversación
# ----------------------------
messages = []

# ----------------------------
# 3️⃣ Definir la función de chat para Gradio
# ----------------------------
def chat_ecommerce_gradio(user_input, history):
    """
    Función compatible con Gradio.
    user_input: texto del usuario
    history: historial del chat [(usuario, asistente), ...]
    """
    # Añadir la nueva entrada del usuario al historial interno
    messages.append(HumanMessage(content=user_input))

    # Ejecutar el agente con el historial completo
    result = agent.invoke({"messages": messages})

    # Actualizar los mensajes guardando el estado
    messages.clear()
    messages.extend(result["messages"])

    # Última respuesta del modelo
    last_ai = result["messages"][-1]
    response = last_ai.content

    # Devolver nueva respuesta + historial actualizado
    history.append((user_input, response))
    return history, history

# ----------------------------
# 4️⃣ Interfaz Gradio
# ----------------------------
with gr.Blocks(title="Agente Ecommerce - LangChain") as demo:
    gr.Markdown("## 🛍️ Asistente de Compras IA para Ecommerce (LangChain + Chroma + OpenAI)")

    chatbot = gr.Chatbot(height=400)
    user_input = gr.Textbox(placeholder="Escribe tu mensaje aquí...", label="Tu mensaje")
    clear_btn = gr.Button("🧹 Limpiar conversación")

    # Conectar las acciones
    user_input.submit(chat_ecommerce_gradio, [user_input, chatbot], [chatbot, chatbot])
    clear_btn.click(lambda: ([], []), None, [chatbot])

# ----------------------------
# 5️⃣ Ejecutar la app
# ----------------------------
demo.launch(share=True)  # share=True genera un link público temporal


/tmp/ipykernel_7790/3790775926.py:48: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_7790/3790775926.py:48: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b2c80abb6b168efb2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
